# Multi-seed Training: RoBERTa + Full (E+K) Index

Trains RoBERTa with the full index on IHC, ISHate and Vicomtech for a single seed. Run multiple times with different `TRAINING_SEED` values to report mean ± std over seeds.

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, str(Path("..").resolve()))
from retriever import encode, retrieve_top_k_above_threshold

## 2. Configuration

Set the training seed via the `TRAINING_SEED` environment variable. The data split seed stays fixed at 42 across all runs.

In [2]:
# seed — set via TRAINING_SEED env var, defaults to 42
SEED = int(os.environ.get('TRAINING_SEED', '42'))
print(f'Training seed: {SEED}')

Training seed: 0


In [3]:
# fixed config
INDEX_DIR       = Path('../..') / 'corpus' / 'index'
WEIGHTS_DIR     = Path('../..') / 'weigths' / 'weights_rac_multiseed'
RESULTS_FILE    = Path('../..') / 'new_results' / f'multiseed_results_s{SEED}.json'
CHUNKS_DIR      = Path('../..') / 'corpus' / 'chunks'

MODEL_KEY       = 'roberta'
MODEL_HF_ID     = 'roberta-base'
INDEX_TYPE      = 'full'          # E+K
RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# Best HP for RoBERTa from tuning (IHC, full index)
K             = 3
THRESHOLD     = 0.4
MAX_K         = 3
MIN_THRESHOLD = 0.4

MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

print(f'Model      : {MODEL_HF_ID}')
print(f'Index      : {INDEX_TYPE}')
print(f'K={K}, threshold={THRESHOLD}')
print(f'Output     : {RESULTS_FILE}')

Model      : roberta-base
Index      : full
K=3, threshold=0.4
Output     : ../../new_results/multiseed_results_s0.json


In [4]:
from training_utils import compute_metrics, tokenize_augmented, filter_records, strip_label_prefix, set_seed

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 3. Load Datasets

Load all three datasets using `data_loaders.py`.

In [5]:
from data_loaders import load_ihc_binary, load_ishate_binary, load_vicomtech

# Data split seed is hardcoded at 42 — same test set across all training seeds
train_ihc, test_ihc     = load_ihc_binary(seed=42)
train_ishate, test_ishate = load_ishate_binary()
vicomtech_train, vicomtech_test = load_vicomtech()

DATASETS = {
    'IHC':       {'train': train_ihc,       'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'train': train_ishate,     'test': test_ishate,    'text_col': 'text'},
    'Vicomtech': {'train': vicomtech_train,  'test': vicomtech_test, 'text_col': 'text'},
}

print(f'IHC        — train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate     — train: {len(train_ishate):,}  test: {len(test_ishate):,}')
print(f'Vicomtech  — train: {len(vicomtech_train):,}  test: {len(vicomtech_test):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

IHC        — train: 19,332  test: 2,148
ISHate     — train: 55,023  test: 4,368
Vicomtech  — train: 1,914  test: 478


## 4. Self-Exclusion Lookup

`chunks_example.csv` maps tweet text → `chunk_id`. Passed at train time so a tweet never retrieves itself as a neighbor.

In [6]:
# build text → chunk_id lookup for self-exclusion at train time
chunks_df = pd.read_csv(CHUNKS_DIR / 'chunks_example.csv')

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 67,864 entries


## 5. Augmentation Helpers

Augment once at MAX_K / MIN_THRESHOLD and cache. `filter_records` applies the model-specific (K, threshold) without re-encoding.

In [7]:
# augmentation helpers
def augment_split_cached(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    """Augment once at MAX_K/MIN_THRESHOLD, storing (text, score) pairs for later filtering."""
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, MIN_THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=MAX_K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': neighbors,  # list of (text, score)
            'label':     example['label'],
        })
    return records

# filter_records, tokenize_augmented, compute_metrics imported from training_utils above

## 6. Load Retriever and Index

In [8]:
# load SBERT retriever and the full index
print(f'Loading retriever: {RETRIEVER_HF_ID} ...')
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f'Retriever ready on {device}')

index_path    = INDEX_DIR / f'vdb_{INDEX_TYPE}.faiss'
ret_index     = faiss.read_index(str(index_path))
print(f'FAISS index loaded: {ret_index.ntotal:,} vectors')

with open(INDEX_DIR / f'lookup_{INDEX_TYPE}.json') as f:
    ret_documents = json.load(f)

Loading retriever: sentence-transformers/all-mpnet-base-v2 ...


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Retriever ready on cuda


FAISS index loaded: 108,816 vectors


In [ ]:
# augment all datasets (cached at MAX_K / MIN_THRESHOLD)
aug_data_cached = {}
for ds_name, ds_cfg in DATASETS.items():
    print(f'\n=== Augmenting {ds_name} ===')
    aug_data_cached[ds_name] = {
        'train': augment_split_cached(ds_cfg['train'], ds_cfg['text_col'], True,
                                      ret_model, ret_tokenizer, ret_index, ret_documents),
        'test':  augment_split_cached(ds_cfg['test'],  ds_cfg['text_col'], False,
                                      ret_model, ret_tokenizer, ret_index, ret_documents),
    }

## 7. Training Loop

Train and evaluate on each dataset for the given seed. Weights saved per dataset.

In [10]:
# train and evaluate on each dataset
results = {}

for ds_name in DATASETS:
    set_seed(SEED)  # reset before each dataset for reproducibility
    print(f"\n{'='*60}")
    print(f'Dataset: {ds_name}  |  Model: {MODEL_KEY}  |  Seed: {SEED}')
    print(f"{'='*60}")

    filtered_train = filter_records(aug_data_cached[ds_name]['train'], K, THRESHOLD)
    filtered_test  = filter_records(aug_data_cached[ds_name]['test'],  K, THRESHOLD)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
    tok_train = tokenize_augmented(filtered_train, tokenizer)
    tok_test  = tokenize_augmented(filtered_test,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_HF_ID, num_labels=2)

    save_path = str(WEIGHTS_DIR / MODEL_KEY / 'sbert' / INDEX_TYPE / ds_name / f'seed{SEED}')
    os.makedirs(save_path, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(Path('../..') / 'checkpoints_rac_multiseed' / MODEL_KEY / INDEX_TYPE / ds_name / f'seed{SEED}'),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='epoch',
        report_to='none',
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_test,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'  Weights saved → {save_path}')

    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = [r['label'] for r in filtered_test]
    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[ds_name] = {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()


Dataset: IHC  |  Model: roberta  |  Seed: 0


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

              precision    recall  f1-score   support

      Non-HS       0.93      0.90      0.92      2681
          HS       0.85      0.90      0.87      1687

    accuracy                           0.90      4368
   macro avg       0.89      0.90      0.90      4368
weighted avg       0.90      0.90      0.90      4368


Dataset: Vicomtech  |  Model: roberta  |  Seed: 0


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.548500,0.499944,0.696343,0.788114,0.715481
2,0.413000,0.544849,0.735970,0.809916,0.748954
3,0.303900,0.456878,0.803852,0.815657,0.805439


  Weights saved → ../../weigths/weights_rac_multiseed/roberta/sbert/full/Vicomtech/seed0


              precision    recall  f1-score   support

      Non-HS       0.87      0.72      0.79       239
          HS       0.76      0.90      0.82       239

    accuracy                           0.81       478
   macro avg       0.82      0.81      0.80       478
weighted avg       0.82      0.81      0.80       478



## 8. Save and Display Results

In [11]:
# save results to JSON
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'seed':    SEED,
    'model':   MODEL_KEY,
    'index':   INDEX_TYPE,
    'results': results,
}
with open(RESULTS_FILE, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Results saved to {RESULTS_FILE}')

Results saved to ../../new_results/multiseed_results_s0.json


In [12]:
# display summary table
rows = []
for ds_name, vals in results.items():
    rows.append({
        'Dataset':   ds_name,
        'F1':        round(vals['macro_f1'], 3),
        'Precision': round(vals['macro_p'],  3),
        'Recall':    round(vals['macro_r'],  3),
    })

df = pd.DataFrame(rows).set_index('Dataset')
print(f'\nRoBERTa (E+K) — seed {SEED}')
display(df)


RoBERTa (E+K) — seed 0


,F1,Precision,Recall
Dataset,,,
IHC,0.806,0.815,0.800
ISHate,0.896,0.893,0.900
Vicomtech,0.804,0.816,0.805
